# GAN Example 2 — Conditional GAN (choose which digit to generate)

The basic DCGAN generates *random* digits — you can't ask for a specific one. A
**Conditional GAN (cGAN)** (Mirza & Osindero, 2014) fixes that by feeding the **class
label** into both networks, so you can request "draw me a 7".

## What changes vs. a plain GAN?

Both networks receive the label as an extra input:

- **Generator** input = noise vector **+** one-hot label → it learns to draw the
  requested digit.
- **Discriminator** input = image **+** one-hot label → it judges "is this a real image
  *of that digit*?". This forces the generator not just to look real, but to **match the
  requested class**.

To keep training light we use a compact **MLP** (dense-layer) GAN rather than the
convolutional DCGAN — it trains quickly on CPU and the conditioning idea is identical.

> Tested with `tensorflow==2.1`. Trains on CPU; a GPU helps.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

tf.random.set_seed(0)
print('TensorFlow', tf.__version__)

## 1. Data

We keep the **labels** this time (the whole point of conditioning). Images are flattened
to 784-vectors and scaled to `[-1, 1]` for the generator's `tanh`.

In [ ]:
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = (x_train.reshape(-1, 784).astype('float32') - 127.5) / 127.5
y_train = y_train.astype('int32')

NUM_CLASSES = 10
NOISE_DIM = 100
BATCH_SIZE = 256

dataset = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
           .shuffle(60000).batch(BATCH_SIZE, drop_remainder=True))
print('Train:', x_train.shape)

## 2. Generator and discriminator (label-conditioned)

Each network simply **concatenates** the one-hot label onto its usual input. Everything
else is a standard MLP.

In [ ]:
def make_generator():
    z = tf.keras.Input((NOISE_DIM,))
    lbl = tf.keras.Input((NUM_CLASSES,))
    h = layers.Concatenate()([z, lbl])
    h = layers.Dense(256)(h); h = layers.LeakyReLU(0.2)(h)
    h = layers.Dense(512)(h); h = layers.LeakyReLU(0.2)(h)
    out = layers.Dense(784, activation='tanh')(h)
    return tf.keras.Model([z, lbl], out)

def make_discriminator():
    img = tf.keras.Input((784,))
    lbl = tf.keras.Input((NUM_CLASSES,))
    h = layers.Concatenate()([img, lbl])
    h = layers.Dense(512)(h); h = layers.LeakyReLU(0.2)(h); h = layers.Dropout(0.3)(h)
    h = layers.Dense(256)(h); h = layers.LeakyReLU(0.2)(h); h = layers.Dropout(0.3)(h)
    out = layers.Dense(1)(h)
    return tf.keras.Model([img, lbl], out)

generator = make_generator()
discriminator = make_discriminator()
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
g_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
d_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

## 3. Training step

Same adversarial logic as the DCGAN, but the **real labels** are threaded through both
networks. The generator is asked to produce images for the *same* labels as the real
batch, so the discriminator's class-aware feedback teaches it the right shapes.

In [ ]:
@tf.function
def train_step(real_imgs, labels):
    bsz = tf.shape(real_imgs)[0]
    onehot = tf.one_hot(labels, NUM_CLASSES)
    noise = tf.random.normal([bsz, NOISE_DIM])

    with tf.GradientTape() as gt, tf.GradientTape() as dt:
        fake = generator([noise, onehot], training=True)
        real_out = discriminator([real_imgs, onehot], training=True)
        fake_out = discriminator([fake, onehot], training=True)
        d_loss = (cross_entropy(tf.ones_like(real_out), real_out) +
                  cross_entropy(tf.zeros_like(fake_out), fake_out))
        g_loss = cross_entropy(tf.ones_like(fake_out), fake_out)

    d_opt.apply_gradients(zip(dt.gradient(d_loss, discriminator.trainable_variables),
                              discriminator.trainable_variables))
    g_opt.apply_gradients(zip(gt.gradient(g_loss, generator.trainable_variables),
                              generator.trainable_variables))
    return g_loss, d_loss

In [ ]:
def show_each_digit(epoch=None):
    labels = tf.one_hot(tf.range(10), NUM_CLASSES)
    noise = tf.random.normal([10, NOISE_DIM])
    imgs = generator([noise, labels], training=False).numpy().reshape(-1, 28, 28)
    plt.figure(figsize=(12, 1.6))
    for i in range(10):
        plt.subplot(1, 10, i + 1)
        plt.imshow(imgs[i] * 0.5 + 0.5, cmap='gray'); plt.axis('off')
        plt.title(str(i))
    if epoch is not None:
        plt.suptitle(f'Epoch {epoch} — one sample per requested digit')
    plt.show()

## 4. Train

Every few epochs we ask the generator for one sample of **each** digit 0–9, so you can
watch the requested classes become legible.

In [ ]:
EPOCHS = 30
for epoch in range(1, EPOCHS + 1):
    for imgs, labels in dataset:
        g_loss, d_loss = train_step(imgs, labels)
    print(f'Epoch {epoch:3d}/{EPOCHS}  gen_loss={float(g_loss):.3f}  disc_loss={float(d_loss):.3f}')
    if epoch % 5 == 0 or epoch == 1:
        show_each_digit(epoch)

## 5. Generate on demand

Pick any digit and generate a row of varied samples of just that class.

In [ ]:
WANT = 7  # <-- change me
labels = tf.one_hot([WANT] * 10, NUM_CLASSES)
imgs = generator([tf.random.normal([10, NOISE_DIM]), labels], training=False).numpy().reshape(-1, 28, 28)
plt.figure(figsize=(12, 1.6))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(imgs[i] * 0.5 + 0.5, cmap='gray'); plt.axis('off')
plt.suptitle(f'Ten generated "{WANT}"s')
plt.show()

### Things to try

- Swap the one-hot concat for a learned `Embedding` layer.
- Make it convolutional (DCGAN-style) for sharper digits.
- Condition on something other than the class — e.g. stroke thickness.
- Compare variety within one class against the unconditional GAN.